In [1]:
import polars as pl
from pybiomart import Server
import os
import numpy as np
import pandas as pd

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2'
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/dataset'

# Data Loading

## Loading the Dataset

In [3]:
# Open file
df = pl.read_csv(os.path.join(DATASET_DIR, 'GSM3718064_HepG2_exp.txt'), separator="\t")

## Split the locus column into chromosome, start, and end

In [4]:
# Split locus column into 3 columns: chr, start, end
df = df.with_columns([
    pl.col("locus").str.split(":").list.get(0).alias("chromosome"),
    pl.col("locus").str.split(":").list.get(1).str.split_exact("-", 1).struct.rename_fields(["start", "end"]).alias("position")
])

# Unnest the column and convert into integer
df = df.unnest("position")

df = df.with_columns([
    pl.col("start").cast(pl.Int64),
    pl.col("end").cast(pl.Int64)
])

df = df.with_columns([
    pl.col('chromosome').str.replace('chr', '').alias('chr')
])

In [5]:
df

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chromosome,start,end,chr
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,i64,i64,str
"""XLOC_000001""","""XLOC_000001""","""OR4F5""","""chr1:69090-70008""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",69090,70008,"""1"""
"""XLOC_000002""","""XLOC_000002""","""LOC100132062,LOC100133331""","""chr1:323891-328581""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0473925,0.0341591,-0.472389,0.0,1.0,1.0,"""no""","""chr1""",323891,328581,"""1"""
"""XLOC_000003""","""XLOC_000003""","""OR4F29""","""chr1:367658-368597""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",367658,368597,"""1"""
"""XLOC_000004""","""XLOC_000004""","""LOC643837""","""chr1:761585-794889""","""hepg2_hr2""","""hepg2_hr3""","""OK""",2.46857,2.8058,0.184736,0.455074,0.63585,0.999565,"""no""","""chr1""",761585,794889,"""1"""
"""XLOC_000005""","""XLOC_000005""","""-""","""chr1:840263-843900""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",840263,843900,"""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030028""","""XLOC_030028""","""-""","""chrY:13319431-13324829""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.337879,0.457897,0.438516,1.13793,0.25685,0.999565,"""no""","""chrY""",13319431,13324829,"""Y"""
"""XLOC_030029""","""XLOC_030029""","""-""","""chrY:13325034-13326120""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.396888,0.357658,-0.150148,0.0,1.0,1.0,"""no""","""chrY""",13325034,13326120,"""Y"""
"""XLOC_030030""","""XLOC_030030""","""-""","""chrY:13331089-13332358""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.477227,0.127331,-1.90609,-1.86192,0.1249,0.999565,"""no""","""chrY""",13331089,13332358,"""Y"""


In [9]:
df.group_by("chromosome").len().sort("len", descending=True)

chromosome,len
str,u32
"""chr1""",2867
"""chr2""",1880
"""chr19""",1711
"""chr11""",1618
"""chr17""",1581
…,…
"""chr17_gl000205_random""",1
"""chrUn_gl000219""",1
"""chrUn_gl000212""",1


## Explode multiple gene names into rows

In [6]:
exploded_df = df.with_columns([
    pl.col("gene").str.split(",")
]).explode("gene")

exploded_df

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chromosome,start,end,chr
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,i64,i64,str
"""XLOC_000001""","""XLOC_000001""","""OR4F5""","""chr1:69090-70008""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",69090,70008,"""1"""
"""XLOC_000002""","""XLOC_000002""","""LOC100132062""","""chr1:323891-328581""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0473925,0.0341591,-0.472389,0.0,1.0,1.0,"""no""","""chr1""",323891,328581,"""1"""
"""XLOC_000002""","""XLOC_000002""","""LOC100133331""","""chr1:323891-328581""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0473925,0.0341591,-0.472389,0.0,1.0,1.0,"""no""","""chr1""",323891,328581,"""1"""
"""XLOC_000003""","""XLOC_000003""","""OR4F29""","""chr1:367658-368597""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",367658,368597,"""1"""
"""XLOC_000004""","""XLOC_000004""","""LOC643837""","""chr1:761585-794889""","""hepg2_hr2""","""hepg2_hr3""","""OK""",2.46857,2.8058,0.184736,0.455074,0.63585,0.999565,"""no""","""chr1""",761585,794889,"""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030028""","""XLOC_030028""","""-""","""chrY:13319431-13324829""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.337879,0.457897,0.438516,1.13793,0.25685,0.999565,"""no""","""chrY""",13319431,13324829,"""Y"""
"""XLOC_030029""","""XLOC_030029""","""-""","""chrY:13325034-13326120""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.396888,0.357658,-0.150148,0.0,1.0,1.0,"""no""","""chrY""",13325034,13326120,"""Y"""
"""XLOC_030030""","""XLOC_030030""","""-""","""chrY:13331089-13332358""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.477227,0.127331,-1.90609,-1.86192,0.1249,0.999565,"""no""","""chrY""",13331089,13332358,"""Y"""


## Saving the original data and exploded data

In [7]:
df.write_csv(os.path.join(WORKING_DIR, 'dataset', 'GSM3718064_HepG2_exp.csv'))
exploded_df.write_csv(os.path.join(WORKING_DIR, 'dataset', 'GSM3718064_HepG2_exp_exploded_gene_name.csv'))

# Data Exploration

## Check the status column

In [8]:
df.group_by(pl.col("status")).len()

status,len
str,u32
"""OK""",13934
"""HIDATA""",1
"""NOTEST""",16092
"""FAIL""",5


## Check Unique Values by start, end, chr, gene, value_1, and  value_2

In [9]:
unique_loc = df.unique(subset=['start', 'end', 'chr', 'gene', 'value_1', 'value_2'])

In [48]:
unique_loc.sort(['test_id'])

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chromosome,start,end,chr
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,i64,i64,str
"""XLOC_000001""","""XLOC_000001""","""OR4F5""","""chr1:69090-70008""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",69090,70008,"""1"""
"""XLOC_000002""","""XLOC_000002""","""LOC100132062,LOC100133331""","""chr1:323891-328581""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0473925,0.0341591,-0.472389,0.0,1.0,1.0,"""no""","""chr1""",323891,328581,"""1"""
"""XLOC_000003""","""XLOC_000003""","""OR4F29""","""chr1:367658-368597""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",367658,368597,"""1"""
"""XLOC_000004""","""XLOC_000004""","""LOC643837""","""chr1:761585-794889""","""hepg2_hr2""","""hepg2_hr3""","""OK""",2.46857,2.8058,0.184736,0.455074,0.63585,0.999565,"""no""","""chr1""",761585,794889,"""1"""
"""XLOC_000005""","""XLOC_000005""","""-""","""chr1:840263-843900""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",840263,843900,"""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030028""","""XLOC_030028""","""-""","""chrY:13319431-13324829""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.337879,0.457897,0.438516,1.13793,0.25685,0.999565,"""no""","""chrY""",13319431,13324829,"""Y"""
"""XLOC_030029""","""XLOC_030029""","""-""","""chrY:13325034-13326120""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.396888,0.357658,-0.150148,0.0,1.0,1.0,"""no""","""chrY""",13325034,13326120,"""Y"""
"""XLOC_030030""","""XLOC_030030""","""-""","""chrY:13331089-13332358""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.477227,0.127331,-1.90609,-1.86192,0.1249,0.999565,"""no""","""chrY""",13331089,13332358,"""Y"""


## Multiple gene names in one locus

In [11]:
# Multiple gene values
df.filter(pl.col('gene').str.contains(","))

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chromosome,start,end,chr
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,i64,i64,str
"""XLOC_000002""","""XLOC_000002""","""LOC100132062,LOC100133331""","""chr1:323891-328581""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0473925,0.0341591,-0.472389,0.0,1.0,1.0,"""no""","""chr1""",323891,328581,"""1"""
"""XLOC_000007""","""XLOC_000007""","""KLHL17,PLEKHN1""","""chr1:851135-917473""","""hepg2_hr2""","""hepg2_hr3""","""OK""",4.04743,4.16567,0.041543,0.0707864,0.94475,0.999565,"""no""","""chr1""",851135,917473,"""1"""
"""XLOC_000012""","""XLOC_000012""","""MIR200A,MIR200B,MIR429""","""chr1:1058324-1105717""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.429365,0.498118,0.214282,0.0573094,0.86515,0.999565,"""no""","""chr1""",1058324,1105717,"""1"""
"""XLOC_000085""","""XLOC_000085""","""APITD1,APITD1-CORT,CORT""","""chr1:10489915-10513512""","""hepg2_hr2""","""hepg2_hr3""","""OK""",10.6686,10.6521,-0.002242,-0.00966,0.9908,0.999565,"""no""","""chr1""",10489915,10513512,"""1"""
"""XLOC_000111""","""XLOC_000111""","""PRAMEF7,PRAMEF8""","""chr1:12976449-12980568""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",12976449,12980568,"""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_029992""","""XLOC_029992""","""NCRNA00230A,NCRNA00230B""","""chrY:19687040-19691353""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chrY""",19687040,19691353,"""Y"""
"""XLOC_029994""","""XLOC_029994""","""CDY2A,CDY2B""","""chrY:19990139-19992100""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chrY""",19990139,19992100,"""Y"""
"""XLOC_029997""","""XLOC_029997""","""HSFY1,HSFY2""","""chrY:20891767-20935621""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chrY""",20891767,20935621,"""Y"""


## Empty gene name

In [12]:
# Empty gene name
df.filter(pl.col('gene') == '-')

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chromosome,start,end,chr
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,i64,i64,str
"""XLOC_000005""","""XLOC_000005""","""-""","""chr1:840263-843900""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr1""",840263,843900,"""1"""
"""XLOC_000010""","""XLOC_000010""","""-""","""chr1:993794-1004436""","""hepg2_hr2""","""hepg2_hr3""","""OK""",1.81732,2.16935,0.255451,0.960974,0.3226,0.999565,"""no""","""chr1""",993794,1004436,"""1"""
"""XLOC_000011""","""XLOC_000011""","""-""","""chr1:1004537-1006002""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0689488,inf,0.0,1.0,1.0,"""no""","""chr1""",1004537,1006002,"""1"""
"""XLOC_000013""","""XLOC_000013""","""-""","""chr1:1058324-1105717""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0987964,0.0987394,-0.000832,0.0,1.0,1.0,"""no""","""chr1""",1058324,1105717,"""1"""
"""XLOC_000016""","""XLOC_000016""","""-""","""chr1:1176090-1177128""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.10094,0.122187,0.275594,0.0,1.0,1.0,"""no""","""chr1""",1176090,1177128,"""1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030028""","""XLOC_030028""","""-""","""chrY:13319431-13324829""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.337879,0.457897,0.438516,1.13793,0.25685,0.999565,"""no""","""chrY""",13319431,13324829,"""Y"""
"""XLOC_030029""","""XLOC_030029""","""-""","""chrY:13325034-13326120""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.396888,0.357658,-0.150148,0.0,1.0,1.0,"""no""","""chrY""",13325034,13326120,"""Y"""
"""XLOC_030030""","""XLOC_030030""","""-""","""chrY:13331089-13332358""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.477227,0.127331,-1.90609,-1.86192,0.1249,0.999565,"""no""","""chrY""",13331089,13332358,"""Y"""


## Single locus, multiple test_id

In [13]:
# Single locus, multiple genes
df.group_by('locus')\
.agg(pl.len().alias('count'))\
.sort('count', descending=True)

locus,count
str,u32
"""chr15:25226925-25367623""",25
"""chr10:9002423-9312354""",19
"""chr21:45904886-46131495""",18
"""chr16:85202966-85722588""",16
"""chr11:1490684-1785501""",16
…,…
"""chr6:169494202-169495587""",1
"""chrX:150336693-150336798""",1
"""chr17:33700942-33701706""",1


In [14]:
df.group_by('locus')\
.agg(pl.len().alias('count'))\
.sort('count', descending=True)\
.filter(pl.col('count') > 1)

locus,count
str,u32
"""chr15:25226925-25367623""",25
"""chr10:9002423-9312354""",19
"""chr21:45904886-46131495""",18
"""chr11:1490684-1785501""",16
"""chr16:85202966-85722588""",16
…,…
"""chr3:193675160-193721448""",2
"""chr4:83550689-83720010""",2
"""chr14:24407939-24520580""",2


In [15]:
df.filter(pl.col('locus') == 'chr15:25226925-25367623')

test_id,gene_id,gene,locus,sample_1,sample_2,status,value_1,value_2,log2(fold_change),test_stat,p_value,q_value,significant,chromosome,start,end,chr
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,str,str,i64,i64,str
"""XLOC_008650""","""XLOC_008650""","""IPW,PAR-SN,PAR5,SNORD109B,SNOR…","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""OK""",25.6997,27.7764,0.112105,0.0105127,0.7459,0.999565,"""no""","""chr15""",25226925,25367623,"""15"""
"""XLOC_008651""","""XLOC_008651""","""SNORD116-1""","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr15""",25226925,25367623,"""15"""
"""XLOC_008652""","""XLOC_008652""","""SNORD116-2""","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr15""",25226925,25367623,"""15"""
"""XLOC_008653""","""XLOC_008653""","""SNORD116-3""","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr15""",25226925,25367623,"""15"""
"""XLOC_008654""","""XLOC_008654""","""SNORD116-7""","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr15""",25226925,25367623,"""15"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_008670""","""XLOC_008670""","""SNORD116-26""","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""OK""",0.0,204.395,inf,NaN,0.2341,0.999565,"""no""","""chr15""",25226925,25367623,"""15"""
"""XLOC_008671""","""XLOC_008671""","""SNORD116-27""","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr15""",25226925,25367623,"""15"""
"""XLOC_008672""","""XLOC_008672""","""SNORD116-28""","""chr15:25226925-25367623""","""hepg2_hr2""","""hepg2_hr3""","""NOTEST""",0.0,0.0,0.0,0.0,1.0,1.0,"""no""","""chr15""",25226925,25367623,"""15"""


# User defined function

## Query all GRCh37/hg19

In [16]:
def query_all():
    # Connect to the GRCh37 archive server
    server = Server(host='http://grch37.ensembl.org')
    
    # Select the human dataset
    dataset = (server.marts['ENSEMBL_MART_ENSEMBL']
               .datasets['hsapiens_gene_ensembl'])

    result = dataset.query(attributes=['chromosome_name', 
                                   'start_position', 
                                   'end_position', 
                                   'strand',
                                   'external_gene_name', 
                                   'ensembl_gene_id',
                                   'gene_biotype'])
    
    result['tss'] = result.apply(lambda row: row['Gene start (bp)'] if row['Strand'] == 1 else row['Gene end (bp)'], axis=1)
    result['tss_2kb_start'] = result['tss'] - 2000
    result['tss_2kb_end'] = result['tss'] + 2000
    return result

In [17]:
all_results = query_all()

## Query by location

In [27]:
def query_ensembl_by_location(item):

    chrom = item[17]
    start = item[15]
    end = item[16]
    gene_name = item[2]
    test_id = item[0]
    locus = item[3]
    
    # Connect to the GRCh37 archive server
    server = Server(host='http://grch37.ensembl.org')
    
    # Select the human dataset
    dataset = (server.marts['ENSEMBL_MART_ENSEMBL']
               .datasets['hsapiens_gene_ensembl'])

    result = dataset.query(attributes=['chromosome_name', 
                                   'start_position', 
                                   'end_position', 
                                   'strand',
                                   'external_gene_name', 
                                   'ensembl_gene_id',
                                   'gene_biotype'],
                       filters={'chromosome_name': chrom,
                                'start': start,
                                'end': end})
    
    result['tss'] = result.apply(lambda row: row['Gene start (bp)'] if row['Strand'] == 1 else row['Gene end (bp)'], axis=1)
    result['tss -2kb'] = result['tss'] - 2000
    result['tss +2kb'] = result['tss'] + 2000
    result['test_id'] = test_id
    result['orig_gene_name'] = gene_name
    result['locus'] = locus
    result['orig_start'] = start
    result['orig_end'] = end
    return result

## Convert Dataframe to Numpy Matrix

In [28]:
exploded_np = exploded_df.to_numpy()

## Query testing

In [35]:
# chr1:323891-328581
# chr1:69090-70008
# chr15:25226925-25367623
# chr10:9002423-9312354
# chr21:45904886-46131495
# chr21:46188604-46224474
# chr1:31184123-31199593
# "PRAMEF7,PRAMEF8"	"chr1:12976449-12980568"
# chr1:993794-1004436 -> empty gene name
# chr1:1058324-1105717 	MIR200A,MIR200B,MIR429
# ARHGEF16	chr1:3366055-3402204 XLOC_000046

test_item = exploded_df.filter(pl.col('gene') == 'MIR200A').to_numpy()[0]

result = query_ensembl_by_location(test_item)
result

,Chromosome/scaffold name,Gene start (bp),Gene end (bp),Strand,Gene name,Gene stable ID,Gene type,tss,tss -2kb,tss +2kb,test_id,orig_gene_name,locus,orig_start,orig_end
0,1,1072397,1079436,1,RP11-465B22.5,ENSG00000223823,lincRNA,1072397,1070397,1074397,XLOC_000012,MIR200A,chr1:1058324-1105717,1058324,1105717
1,1,1102484,1102578,1,MIR200B,ENSG00000207730,miRNA,1102484,1100484,1104484,XLOC_000012,MIR200A,chr1:1058324-1105717,1058324,1105717
2,1,1103243,1103332,1,MIR200A,ENSG00000207607,miRNA,1103243,1101243,1105243,XLOC_000012,MIR200A,chr1:1058324-1105717,1058324,1105717
3,1,1104385,1104467,1,MIR429,ENSG00000198976,miRNA,1104385,1102385,1106385,XLOC_000012,MIR200A,chr1:1058324-1105717,1058324,1105717
4,1,1104737,1105723,1,RP11-465B22.8,ENSG00000272141,lincRNA,1104737,1102737,1106737,XLOC_000012,MIR200A,chr1:1058324-1105717,1058324,1105717


In [47]:
result.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Chromosome/scaffold name  1 non-null      object
 1   Gene start (bp)           1 non-null      int64 
 2   Gene end (bp)             1 non-null      int64 
 3   Strand                    1 non-null      int64 
 4   Gene name                 1 non-null      object
 5   Gene stable ID            1 non-null      object
 6   Gene type                 1 non-null      object
 7   tss                       1 non-null      int64 
 8   tss -2kb                  1 non-null      int64 
 9   tss +2kb                  1 non-null      int64 
 10  test_id                   1 non-null      object
 11  orig_gene_name            1 non-null      object
 12  locus                     1 non-null      object
 13  orig_start                1 non-null      int64 
 14  orig_end                  1 no

In [37]:
exploded_np[0]

array(['XLOC_000001', 'XLOC_000001', 'OR4F5', 'chr1:69090-70008',
       'hepg2_hr2', 'hepg2_hr3', 'NOTEST', 0.0, 0.0, 0.0, 0.0, 1.0, 1.0,
       'no', 'chr1', 69090, 70008, '1'], dtype=object)

# Query for all items

In [42]:
output_path = os.path.join(WORKING_DIR, 'dataset', 'ensembl.csv')

i = 1
total = exploded_np.shape[0]

for item in exploded_np:

    if ((i % 100 == 0) or (i == total)):
        print(f"{i}/{total} - {item[0]} - {item[2]}")
    
    result = query_ensembl_by_location(item)
    result.to_csv(output_path, mode='a', header=(not os.path.exists(output_path)), index=False)
    i += 1

100/31121 - XLOC_000094 - AGTRAP
200/31121 - XLOC_000190 - RPL11
300/31121 - XLOC_000284 - PHC2
400/31121 - XLOC_000380 - LOC400752
500/31121 - XLOC_000477 - HHLA3
600/31121 - XLOC_000569 - HEJ1
700/31121 - XLOC_000667 - -
800/31121 - XLOC_000761 - LCE3C
900/31121 - XLOC_000855 - PYHIN1
1000/31121 - XLOC_000951 - GPR52
1100/31121 - XLOC_001050 - -
1200/31121 - XLOC_001149 - EPHX1
1300/31121 - XLOC_001246 - OR2T5
1400/31121 - XLOC_001341 - -
1500/31121 - XLOC_001435 - LUZP1
1600/31121 - XLOC_001531 - -
1700/31121 - XLOC_001627 - STIL
1800/31121 - XLOC_001726 - MCOLN3
1900/31121 - XLOC_001823 - SLC16A1
2000/31121 - XLOC_001915 - ANP32E
2100/31121 - XLOC_002007 - C1orf104
2200/31121 - XLOC_002101 - LMX1A
2300/31121 - XLOC_002199 - PDC
2400/31121 - XLOC_002296 - RD3
2500/31121 - XLOC_002393 - TOMM20
2600/31121 - XLOC_002492 - -
2700/31121 - XLOC_002592 - -
2800/31121 - XLOC_002692 - -
2900/31121 - XLOC_002792 - -
3000/31121 - XLOC_002892 - ANKRD16
3100/31121 - XLOC_002990 - RET
3200/31121 